In [12]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

MODEL = "gpt-4.1-mini"
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": "what is RAG?"
        }
    ]
)

print(response.choices[0].message.content)

RAG can refer to different things depending on the context, but some common meanings include:

1. **RAG (Red-Amber-Green) Status**:  
   Often used in project management and reporting to indicate the status of a task or project:
   - **Red**: Problematic or behind schedule
   - **Amber (or Yellow)**: At risk or needs attention
   - **Green**: On track or progressing well

2. **RAG (Retrieval-Augmented Generation)**:  
   In the context of natural language processing (NLP) and AI, RAG is a method combining retrieval-based and generative models. It incorporates a retrieval system to fetch relevant information from a knowledge base or documents and uses a generative model (like a Transformer-based language model) to produce answers or text based on that retrieved information. This improves the accuracy and informativeness of generated responses, especially for knowledge-intensive tasks.

3. **Other usages**:  
   - A "rag" as a cloth or piece of fabric  
   - In university contexts (espec

In [13]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": "what is RAG in Natural Language Processing?"
        }
    ]
)

print(response.choices[0].message.content)

In Natural Language Processing (NLP), **RAG** stands for **Retrieval-Augmented Generation**. It is a hybrid model that combines two key components:

1. **Retrieval**: The model first retrieves relevant documents or pieces of information from a large external knowledge base or corpus.
2. **Generation**: Then, it uses a generative model (typically based on transformers) to produce a response or output, conditioning its generation on the retrieved information.

### Why RAG?

Traditional generative models like GPT can generate fluent text but sometimes struggle to provide accurate or up-to-date facts if they are not part of the training data. On the other hand, retrieval models excel at bringing in relevant facts from large external sources but don't generate text on their own.

RAG models **augment generation with retrieval** by grounding the output in real factual data, leading to more accurate, informative, and contextually relevant responses. This approach is particularly beneficial in

In [14]:
questions = [
    "what is overfitting in machine learning?",
    "Explain gradient descent in simple terms.",
    "what is neural network?",
    "Difference between supervised and unsupervised learning?",
    "Explain backpropagation in neural networks."
]

In [15]:
PROMT_V1 = """
Answer the following student questions clearly.
Question: (Question)
"""

In [22]:
import time
import pandas as pd

def evaluate_prompt(
        questions,
        prompt_template,
        prompt_version,
        model
):
    results = []

    for idx, question in enumerate(questions, start=1):

        prompt = prompt_template.format(
            question=question
        )

        start_time = time.time()

        response = client.chat.completions.create(
            model=model,
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        )

        end_time = time.time()

        latency = end_time - start_time

        results.append(
            {
                "question_id": idx,
                "prompt_version": prompt_version,
                "model": "gpt-4.1-mini",
                "latency_seconds": latency,
                "input_tokens": response.usage.prompt_tokens,
                "output_tokens": response.usage.completion_tokens,
                "total_tokens": response.usage.total_tokens,
                "response_status": "success"
            }
        )

    return pd.DataFrame(results)

In [ ]:
df_v1 = evaluate_prompt(
    questions,
    PROMT_V1,
    "v1",
    MODEL
)

In [ ]:
print(df_v1['latency_seconds'].describe())

count    5.000000
mean     1.049411
std      0.353179
min      0.717722
25%      0.824947
50%      0.849229
75%      1.348691
max      1.506465
Name: latency_seconds, dtype: float64


In [ ]:
latency_stats_v1 = df_v1['latency_seconds'].agg(
    ['mean', 'median', "min", 'max', 'std']
).to_frame("value")

print(latency_stats_v1)

           value
mean    1.049411
median  0.849229
min     0.717722
max     1.506465
std     0.353179
